In [1]:
# 1: importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3


In [2]:
# 2: data importing and exploring

try: 
    df = pd.read_csv("../data/superstore.csv", encoding='latin-1')
except FileNotFoundError:
    print("File not found")


# df.head()
# df.isnull().sum()
df.dtypes
# df.describe()
# df.columns
# df.head()

Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

In [ ]:
# 3: data cleaning
df.drop_duplicates(inplace=True)
df['Order Date'] = pd.to_datetime(
            df['Order Date'],
            format='%m/%d/%Y'
        )

df['Order Date'] = df['Order Date'].dt.strftime('%Y-%m-%d')

# print(df['Order Date'].dtype)

df['Ship Date'] = pd.to_datetime(
            df['Ship Date'],
            format='%m/%d/%Y'
    )

df['Ship Date'] = df['Ship Date'].dt.strftime('%Y-%m-%d')

# print(df['Ship Date'].dtype)

# df.dtypes

In [46]:
# SQLite connection
try:
    conn = sqlite3.connect(':memory:')
    print('Connection made')
    df.to_sql('superstore', conn, if_exists='replace', index=False)
    print('BD criada')

except:
    print('Impossivel conectar ao SQLite')



Connection made
BD criada


In [ ]:
# Executes queries
# P.S.: run this section before running any query from the business questions

def execute(query):
    try:
        result = pd.read_sql_query(query, conn)
        return result
    except:
        print('Erro ao ler query')

In [48]:
query = """
        SELECT "Order Date"
        FROM superstore
        LIMIT 5
    """
print(execute(query))

   Order Date
0  2016-11-08
1  2016-11-08
2  2016-06-12
3  2015-10-11
4  2015-10-11


In [115]:
# 4: business questions

# 1. Which region generates the most total sales?
query = """
        SELECT Region, max(total_sales) AS "Total Sales"
        FROM ( 
            SELECT
                Region,
                ROUND(SUM(sales), 2) AS total_sales
            FROM superstore
            GROUP BY region
        ) AS maxi
    """
print(execute(query))


Erro ao ler query
None


In [98]:
# 2. What are the top 5 products by total revenue?
query = """
        SELECT
            "Product Name",
            ROUND(SUM(profit), 2) AS Revenue
        FROM superstore
        GROUP BY "product name"
        ORDER BY revenue DESC
        LIMIT 5
    """
print(execute(query))

Erro ao ler query
None


In [78]:
# 3. What do monthly sales trends look like over time?
query = """
        SELECT
            CASE strftime('%m', datetime("Order Date"))
                WHEN '01' THEN 'January'
                WHEN '02' THEN 'February'
                WHEN '03' THEN 'March'
                WHEN '04' THEN 'April'
                WHEN '05' THEN 'May'
                WHEN '06' THEN 'June'
                WHEN '07' THEN 'July'
                WHEN '08' THEN 'August'
                WHEN '09' THEN 'September'
                WHEN '10' THEN 'October'
                WHEN '11' THEN 'November'
                WHEN '12' THEN 'December'
            END AS Month,
            strftime('%Y', datetime("Order Date")) AS Year,
            ROUND(SUM(Sales), 2) AS Year_Month_Sales
        FROM superstore
        GROUP BY Year, Month
        ORDER BY Year, strftime('%m', datetime("Order Date"))
    """
print(execute(query))

        Month  Year  Year_Month_Sales
0     January  2014          14236.90
1    February  2014           4519.89
2       March  2014          55691.01
3       April  2014          28295.35
4         May  2014          23648.29
5        June  2014          34595.13
6        July  2014          33946.39
7      August  2014          27909.47
8   September  2014          81777.35
9     October  2014          31453.39
10   November  2014          78628.72
11   December  2014          69545.62
12    January  2015          18174.08
13   February  2015          11951.41
14      March  2015          38726.25
15      April  2015          34195.21
16        May  2015          30131.69
17       June  2015          24797.29
18       July  2015          28765.33
19     August  2015          36898.33
20  September  2015          64595.92
21    October  2015          31404.92
22   November  2015          75972.56
23   December  2015          74919.52
24    January  2016          18542.49
25   Februar

In [96]:
# 4. Which category has the best profit margin?
query = """
        SELECT
            Category,
            MAX(Profit_Margin) AS "Best Profit Margin (%)"
        FROM (
            SELECT
                Category,
                ROUND((SUM(Profit) / SUM(Sales)) * 100, 2) AS Profit_Margin
            FROM superstore
            GROUP BY Category
        ) AS PF
    """
print(execute(query))

     Category  Best Profit Margin (%)
0  Technology                    17.4
